# Datathon Task 1 V7 — DirectLR Anchor + Directed Residual-GNN

V7 combines the strongest verified pieces from our V6 and Raphael V2: V6's tuned direct low-rank anchor and Raphael's zero-initialized directed GNN that learns only the anchor residual.


In [ ]:
from __future__ import annotations
import gc, hashlib, importlib.util, json, math, os, random, re, sys, time, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm, trange
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
warnings.filterwarnings('ignore')
try:
    from IPython.display import display
except Exception:
    def display(obj): print(obj.to_string() if hasattr(obj,'to_string') else obj)

SEED=42
random.seed(SEED); np.random.seed(SEED)
HORIZONS=(5,10,15); HISTORY=15; EXPECTED_ROADS=1260
FAST_MODE=os.getenv('DATATON_FAST_MODE','0')=='1'
KAGGLE_DATASET_PATH=Path('/kaggle/input/competitions/datathon-task-1/dataset-task1')
LOCAL_CANDIDATES=[Path.cwd()/'dataset-task1']+[p/'dataset-task1' for p in Path.cwd().parents]
LOCAL_DATASET_PATH=next((p for p in LOCAL_CANDIDATES if p.exists()),LOCAL_CANDIDATES[0])
DATASET_PATH=KAGGLE_DATASET_PATH if KAGGLE_DATASET_PATH.exists() else LOCAL_DATASET_PATH
if not DATASET_PATH.exists(): raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')
IS_KAGGLE=KAGGLE_DATASET_PATH.exists()
WORK_ROOT=Path('/kaggle/working') if IS_KAGGLE else DATASET_PATH.parent/'tmp/jupyter-notebook/v7-smoke'
OUTPUT_DIR=WORK_ROOT; CACHE_DIR=WORK_ROOT/'v7_cache'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True); CACHE_DIR.mkdir(parents=True,exist_ok=True)

def available(package): return importlib.util.find_spec(package) is not None
OPTIONAL={p:available(p) for p in ['matplotlib','seaborn','torch','sklearn']}
print('Using dataset:',DATASET_PATH)
print('Mode:','FAST' if FAST_MODE else 'FULL','| Python:',sys.version.split()[0],'| optional:',OPTIONAL)

def mib(x): return x/2**20
def memory_estimate(shape,dtype=np.float32): return mib(int(np.prod(shape))*np.dtype(dtype).itemsize)
def collect_garbage():
    gc.collect()
    if OPTIONAL['torch']:
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()


## 1. Dataset discovery and structural audit

Nothing below assumes filename suffixes or JSON key shapes. Speed blocks are
sorted by the numeric `m` block and first numeric suffix.


In [ ]:
def natural_key(path: Path):
    return [int(x) if x.isdigit() else x.lower()
            for x in re.split(r"(\d+)", path.name)]


all_files = sorted([p for p in DATASET_PATH.rglob("*") if p.is_file()])
file_manifest = pd.DataFrame({
    "file": [str(p.relative_to(DATASET_PATH)) for p in all_files],
    "size_mib": [mib(p.stat().st_size) for p in all_files],
    "suffix": [p.suffix.lower() for p in all_files],
})
display(file_manifest)

npy_audit = []
for p in sorted(DATASET_PATH.rglob("*.npy"), key=natural_key):
    a = np.load(p, mmap_mode="r")
    npy_audit.append({
        "file": str(p.relative_to(DATASET_PATH)), "shape": a.shape,
        "dtype": str(a.dtype), "mib": mib(a.nbytes),
        "min": float(np.nanmin(a)), "max": float(np.nanmax(a)),
    })
display(pd.DataFrame(npy_audit))

speed_files = sorted((DATASET_PATH / "train").glob("train_speed*.npy"), key=natural_key)
text_files = sorted((DATASET_PATH / "train").glob("train_text*.json"), key=natural_key)
assert speed_files, "No training speed blocks discovered"
train_blocks = [np.load(p, mmap_mode="r") for p in speed_files]
test_X = np.load(DATASET_PATH / "test" / "test_X_hist.npy", mmap_mode="r")
A_raw = np.load(DATASET_PATH / "static" / "matrix.npy", mmap_mode="r")
active_mask = np.load(DATASET_PATH / "static" / "active_mask.npy", mmap_mode="r")
sample_path = (DATASET_PATH / "sample_submission.csv"
               if (DATASET_PATH / "sample_submission.csv").exists()
               else DATASET_PATH / "test" / "sample_submission.csv")
sample = pd.read_csv(sample_path)

R = train_blocks[0].shape[1]
N_TEST = test_X.shape[0]
assert R == EXPECTED_ROADS
assert all(a.ndim == 2 and a.shape[1] == R for a in train_blocks)
assert test_X.shape[1:] == (HISTORY, R)
assert A_raw.shape == (R, R)
assert len(sample) == N_TEST * len(HORIZONS) * R
assert sample.shape[1] >= 2
print(f"{len(train_blocks)} train blocks:", [a.shape for a in train_blocks])
print("Test:", test_X.shape, "| sample submission:", sample.shape)
print("Estimated full prediction tensor:", memory_estimate((N_TEST, 3, R)), "MiB")


def load_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)


train_text_dicts = [load_json(p) for p in text_files]
test_text_dict = load_json(DATASET_PATH / "test" / "test_texts.json")
roads_raw = load_json(DATASET_PATH / "static" / "Roads1260.json")


def ordered_dict_values(d: dict) -> list:
    def last_number(k):
        m = re.findall(r"\d+", str(k))
        return int(m[-1]) if m else 0
    return [d[k] for k in sorted(d, key=last_number)]


train_texts = [ordered_dict_values(d) for d in train_text_dicts]
test_texts = ordered_dict_values(test_text_dict)
assert len(train_texts) == len(train_blocks)
assert all(len(txt) == len(arr) for txt, arr in zip(train_texts, train_blocks))
assert len(test_texts) == N_TEST
print("JSON audit:")
for p, x in zip(text_files, train_text_dicts):
    first_key = next(iter(x))
    print(p.name, type(x).__name__, len(x), "first:", first_key,
          repr(x[first_key])[:160])
print("Road metadata outer type/length:", type(roads_raw).__name__, len(roads_raw))
print("First road metadata entry:", roads_raw[0])


## 2. Temporal interpretation and data-quality audit

Fifteen input steps represent 60 minutes, so one step is four minutes.
Therefore the stated targets map exactly to 5, 10, and 15 steps. Filenames
contain sequence indices but no wall-clock timestamp; this is reported as a
verified arithmetic interpretation, not evidence of calendar alignment.


In [ ]:
STEP_MINUTES = 60 / HISTORY
print("Minutes per step:", STEP_MINUTES)
assert STEP_MINUTES == 4 and tuple(int(h * STEP_MINUTES) for h in HORIZONS) == (20, 40, 60)


def quality_stats(name: str, a: np.ndarray) -> dict:
    x = np.asarray(a)
    return {
        "name": name, "rows": x.shape[0], "roads": x.shape[-1],
        "nan": int(np.isnan(x).sum()), "inf": int(np.isinf(x).sum()),
        "zero_pct": float((x == 0).mean() * 100),
        "negative": int((x < 0).sum()),
        "mean": float(np.nanmean(x)), "std": float(np.nanstd(x)),
        "q001": float(np.nanquantile(x, .001)),
        "q50": float(np.nanquantile(x, .5)),
        "q999": float(np.nanquantile(x, .999)),
        "max": float(np.nanmax(x)),
    }


quality = [quality_stats(p.name, a) for p, a in zip(speed_files, train_blocks)]
quality.append(quality_stats("test_X_hist.npy", test_X))
display(pd.DataFrame(quality))
assert all(np.isfinite(np.asarray(a)).all() for a in train_blocks)
assert np.isfinite(np.asarray(test_X)).all()

train_concat = np.concatenate([np.asarray(a) for a in train_blocks], axis=0)
global_mean = np.float32(train_concat.mean())
road_mean = train_concat.mean(0).astype(np.float32)
road_std = train_concat.std(0).astype(np.float32)
road_q001 = np.quantile(train_concat, .001, axis=0).astype(np.float32)
road_q999 = np.quantile(train_concat, .999, axis=0).astype(np.float32)
road_audit = pd.DataFrame({
    "road": np.arange(R), "mean": road_mean, "std": road_std,
    "zero_pct": (train_concat == 0).mean(0) * 100,
    "min": train_concat.min(0), "max": train_concat.max(0),
})
print("Constant roads:", int((road_std == 0).sum()))
display(road_audit.describe(percentiles=[.01, .1, .5, .9, .99]).T)

# Duplicate road columns are screened with compact hashes, then exact comparison.
column_hashes = defaultdict(list)
for j in range(R):
    h = hashlib.blake2b(np.ascontiguousarray(train_concat[:, j]).view(np.uint8),
                        digest_size=12).hexdigest()
    column_hashes[h].append(j)
duplicate_road_groups = []
for group in column_hashes.values():
    if len(group) > 1:
        exact = [group[0]] + [j for j in group[1:]
                              if np.array_equal(train_concat[:, group[0]], train_concat[:, j])]
        if len(exact) > 1:
            duplicate_road_groups.append(exact)
print("Exact duplicate road groups:", duplicate_road_groups[:10],
      "| count:", len(duplicate_road_groups))

# Train/test drift summaries. Wasserstein is optional; quantile distance is always available.
drift = pd.DataFrame({
    "road": np.arange(R),
    "train_mean": train_concat.mean(0),
    "test_mean": np.asarray(test_X).mean((0, 1)),
    "train_std": train_concat.std(0),
    "test_std": np.asarray(test_X).std((0, 1)),
})
drift["mean_shift_z"] = ((drift.test_mean - drift.train_mean) /
                         (drift.train_std + 1e-3))
print("Absolute standardized train/test mean shift quantiles:",
      drift.mean_shift_z.abs().quantile([.5, .9, .99]).to_dict())
display(drift.iloc[drift.mean_shift_z.abs().nlargest(10).index])


## 3. Traffic EDA

Plots are concise and optional. The numerical tables are produced even when
plotting libraries are unavailable.


In [ ]:
network_series = [np.asarray(a).mean(1) for a in train_blocks]
eda_rows = []
for b, a in enumerate(train_blocks):
    x = np.asarray(a)
    d = np.diff(x, axis=0)
    eda_rows.append({
        "block": b, "timesteps": len(x), "network_mean": x.mean(),
        "network_std_over_time": x.mean(1).std(),
        "mean_abs_step_change": np.abs(d).mean(),
        "p99_abs_step_change": np.quantile(np.abs(d), .99),
        "duplicated_timesteps": int(pd.DataFrame(x).duplicated().sum()),
    })
display(pd.DataFrame(eda_rows))

if OPTIONAL["matplotlib"]:
    import matplotlib.pyplot as plt
    import seaborn as sns
    fig, axes = plt.subplots(2, 2, figsize=(15, 8))
    axes[0, 0].hist(train_concat.ravel()[::50], bins=80)
    axes[0, 0].set(title="Overall speed distribution", xlabel="speed")
    for b, s in enumerate(network_series):
        axes[0, 1].plot(s, lw=.7, label=f"block {b}")
    axes[0, 1].set(title="Average network speed over time")
    axes[0, 1].legend()
    axes[1, 0].hist(road_std, bins=50)
    axes[1, 0].set(title="Road volatility")
    show = np.asarray(train_blocks[0])[:min(720, len(train_blocks[0])), ::20].T
    sns.heatmap(show, cmap="viridis", ax=axes[1, 1], cbar_kws={"label": "speed"})
    axes[1, 1].set(title="Block 0 heatmap (every 20th road)")
    plt.tight_layout()
    plt.show()

    representative = [
        int(np.argmax(road_mean)), int(np.argmin(road_mean)),
        int(np.argmax(road_std)), int(np.argsort(road_std)[R // 2])
    ]
    fig, axes = plt.subplots(len(representative), 1, figsize=(14, 9), sharex=True)
    for ax, r in zip(axes, representative):
        ax.plot(np.asarray(train_blocks[0])[:720, r], lw=.8)
        ax.set_ylabel(f"r{r}")
    axes[0].set_title("Representative roads: high, congested, volatile, median")
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib unavailable: plot cells skipped without affecting modeling.")

# Autocorrelation of the city factor at plausible intraday lags.
for lag in [1, 5, 10, 15, 60, 180, 360, 720, 2520]:
    vals = []
    for s in network_series:
        if lag < len(s):
            vals.append(np.corrcoef(s[:-lag], s[lag:])[0, 1])
    print(f"city-factor ACF lag {lag:4d} ({lag*4:5d} min):",
          np.round(vals, 4).tolist())


## 4. Directed road-network and geometry analysis


In [ ]:
A = np.asarray(A_raw, dtype=np.float32)
src, dst = np.nonzero(A)
weights = A[src, dst].astype(np.float32)
out_degree = A.sum(1)
in_degree = A.sum(0)
print({
    "shape": A.shape, "edges": int((A != 0).sum()),
    "density": float((A != 0).mean()),
    "weighted": bool(np.any((A != 0) & (A != 1))),
    "symmetric": bool(np.array_equal(A, A.T)),
    "self_loops": int(np.count_nonzero(np.diag(A))),
    "isolated_out": int((out_degree == 0).sum()),
    "isolated_in": int((in_degree == 0).sum()),
})


def flatten_road_metadata(entry) -> dict:
    # Each road can contain multiple link dictionaries.
    links = entry if isinstance(entry, list) else [entry]
    links = [x for x in links if isinstance(x, dict)]
    coords = [x.get("coordList", []) for x in links]
    flat = [v for c in coords for v in c]
    lon = np.asarray(flat[0::2], dtype=float) if flat else np.array([])
    lat = np.asarray(flat[1::2], dtype=float) if flat else np.array([])
    names = [str(x.get("roadName", "")) for x in links if x.get("roadName")]
    return {
        "n_links": len(links),
        "length": sum(float(x.get("length", 0) or 0) for x in links),
        "roadclass": links[0].get("roadclass", -1) if links else -1,
        "formway": links[0].get("formway", -1) if links else -1,
        "road_name": names[0] if names else "",
        "lon": lon.mean() if len(lon) else np.nan,
        "lat": lat.mean() if len(lat) else np.nan,
        "orientation": (math.atan2(lat[-1] - lat[0], lon[-1] - lon[0])
                        if len(lat) > 1 else np.nan),
    }


roads = pd.DataFrame([flatten_road_metadata(x) for x in roads_raw])
roads["road"] = np.arange(R)
roads["out_degree"] = out_degree
roads["in_degree"] = in_degree
display(roads.head())
display(roads.describe(include="all").T)


def graph_aggregate(v: np.ndarray, reverse=False) -> np.ndarray:
    """Sparse directed weighted neighbor mean for a single R-vector."""
    s, d = (dst, src) if reverse else (src, dst)
    num = np.bincount(s, weights=(weights * v[d]), minlength=R)
    den = np.bincount(s, weights=weights, minlength=R)
    return (num / np.maximum(den, 1e-6)).astype(np.float32)


# Verify that graph-neighbor pairs are more correlated than random pairs.
rng = np.random.default_rng(SEED)
probe = train_concat[::max(1, len(train_concat)//2000)]
nonself = src != dst
edge_pairs = np.column_stack([src[nonself], dst[nonself]])
edge_pairs = edge_pairs[rng.choice(len(edge_pairs), min(2000, len(edge_pairs)), replace=False)]
rand_pairs = rng.integers(0, R, size=(len(edge_pairs), 2))


def mean_pair_corr(x, pairs):
    vals = []
    for i, j in pairs:
        if x[:, i].std() > 0 and x[:, j].std() > 0:
            vals.append(np.corrcoef(x[:, i], x[:, j])[0, 1])
    return float(np.nanmean(vals))


print("Mean edge correlation:", mean_pair_corr(probe, edge_pairs))
print("Mean random-pair correlation:", mean_pair_corr(probe, rand_pairs))
del probe

# The 1,296 mask is a 36×36 grid with one inactive cell per row; its 1,260
# active positions map naturally to the road count. We do not silently index the
# roads by it unless this invariant holds.
print("active_mask:", active_mask.shape, int(active_mask.sum()))
if active_mask.size == 36 * 36 and active_mask.sum() == R:
    grid_positions = np.argwhere(np.asarray(active_mask).reshape(36, 36))
    print("Verified 36×36 active grid mapping; first positions:", grid_positions[:5])


## 5. Event-text audit and compact features


In [ ]:
KEYWORDS = {
    "accident": ["accident", "collision", "crash"],
    "closure": ["closure", "closed", "prohibit"],
    "construction": ["construction", "roadwork", "maintenance"],
    "congestion": ["congestion", "traffic jam", "slow traffic"],
    "weather": ["rain", "snow", "fog", "storm", "wind"],
    "severity": ["serious", "major", "severe", "general"],
    "announcement": ["announcement", "notice"],
}


def text_features(text: str) -> np.ndarray:
    t = (text or "").lower()
    tokens = re.findall(r"\b\w+\b", t)
    base = [
        float(not bool(t.strip())), len(t), len(tokens),
        t.count("."), len(re.findall(r"\d+", t)),
    ]
    return np.asarray(base + [
        sum(t.count(k) for k in kws) for kws in KEYWORDS.values()
    ], dtype=np.float32)


text_feature_names = (["empty", "characters", "tokens", "sentences", "numbers"]
                      + list(KEYWORDS))
train_text_features = [np.vstack([text_features(x) for x in block])
                       for block in train_texts]
test_text_features = np.vstack([text_features(x) for x in test_texts])
text_audit = []
for name, texts in [(p.name, x) for p, x in zip(text_files, train_texts)] + [("test", test_texts)]:
    counts = Counter(texts)
    text_audit.append({
        "source": name, "n": len(texts),
        "empty_pct": 100 * sum(not t.strip() for t in texts) / len(texts),
        "unique": len(counts), "duplicate_pct": 100 * (1 - len(counts)/len(texts)),
        "median_chars": float(np.median([len(t) for t in texts])),
    })
display(pd.DataFrame(text_audit))
display(pd.DataFrame(test_text_features, columns=text_feature_names).describe().T)

all_train_text = [t for block in train_texts for t in block]
train_text_lookup = defaultdict(list)
offset = 0
for b, block in enumerate(train_texts):
    for t, text in enumerate(block):
        train_text_lookup[text].append((b, t))
exact_text_hits = [{
    "test_sample": i, "matches": train_text_lookup[text][:10],
    "n_matches": len(train_text_lookup[text])
} for i, text in enumerate(test_texts) if text in train_text_lookup]
print("Exact test/train text matches:", len(exact_text_hits))
display(pd.DataFrame(exact_text_hits).head(20))

# Road-name relevance is conservative: normalize Latin metadata names only,
# because event text is English while metadata may be Chinese.
latin_names = {}
for r, name in enumerate(roads.road_name.fillna("").str.lower()):
    norm = re.sub(r"[^a-z0-9 ]+", " ", name).strip()
    if len(norm) >= 4:
        latin_names[r] = norm
print("Road names directly matchable to English event text:", len(latin_names))


## 6. Leakage and recoverability audit

Hashes use 16 deterministic road probes to propose candidates; every candidate
is verified on all 1,260 roads. This avoids a naïve all-pairs scan.


In [ ]:
PROBE_ROADS = np.array([0, 1, 2, 7, 13, 31, 63, 127, 255, 383, 511,
                        767, 911, 1023, 1187, 1259])


def digest(x: np.ndarray, decimals=None) -> bytes:
    z = np.asarray(x)
    if decimals is not None:
        z = np.round(z, decimals=decimals)
    return hashlib.blake2b(np.ascontiguousarray(z).view(np.uint8),
                           digest_size=16).digest()


def exact_train_matches(test_windows, train_arrays, lengths=(15, 10, 5),
                        decimals_options=(None, 3, 1)):
    rows = []
    from numpy.lib.stride_tricks import sliding_window_view
    for L in lengths:
        for decimals in decimals_options:
            lookup = defaultdict(list)
            for b, arr in enumerate(train_arrays):
                # axis order from sliding_window_view is (window, road, time)
                view = sliding_window_view(
                    np.asarray(arr[:, PROBE_ROADS]), window_shape=L, axis=0
                ).transpose(0, 2, 1)
                for s, w in enumerate(view):
                    lookup[digest(w, decimals)].append((b, s))
            for i, qfull in enumerate(test_windows):
                q = np.asarray(qfull[-L:])
                candidates = lookup.get(digest(q[:, PROBE_ROADS], decimals), [])
                for b, s in candidates:
                    ref = np.asarray(train_arrays[b][s:s+L])
                    equal = (np.array_equal(ref, q) if decimals is None else
                             np.array_equal(np.round(ref, decimals),
                                            np.round(q, decimals)))
                    if equal:
                        end = s + L - 1
                        future_available = end + max(HORIZONS) < len(train_arrays[b])
                        rows.append({
                            "test_sample": i, "best_train_block": b,
                            "best_train_index": s, "length": L,
                            "round_decimals": decimals, "exact_match": decimals is None,
                            "match_mse": float(np.mean((ref - q)**2)),
                            "history_overlap": L,
                            "future_available": future_available,
                            "confidence": "exact" if decimals is None else f"rounded_{decimals}",
                        })
    return pd.DataFrame(rows)


leakage_audit = exact_train_matches(test_X, train_blocks)
print("Verified exact/rounded train-history matches:", len(leakage_audit))
display(leakage_audit.head())


def test_chain_audit(X, shifts=(1, 5, 10)):
    rows = []
    for shift in shifts:
        L = HISTORY - shift
        starts = defaultdict(list)
        for j in range(len(X)):
            starts[digest(np.asarray(X[j, :L, PROBE_ROADS]))].append(j)
        for i in range(len(X)):
            key = digest(np.asarray(X[i, shift:, PROBE_ROADS]))
            for j in starts.get(key, []):
                if i != j and np.array_equal(np.asarray(X[i, shift:]),
                                              np.asarray(X[j, :L])):
                    rows.append({"from": i, "to": j, "shift": shift,
                                 "overlap": L})
    return pd.DataFrame(rows)


test_chains = test_chain_audit(test_X)
print("Exact test-to-test overlap chains:", len(test_chains))
display(test_chains.head())

# Check whether train blocks overlap or continue each other at their boundaries.
block_overlap_rows = []
for i in range(len(train_blocks)):
    for j in range(i + 1, len(train_blocks)):
        for L in [1, 5, 10, 15, 60]:
            eq = np.array_equal(np.asarray(train_blocks[i][-L:]),
                                np.asarray(train_blocks[j][:L]))
            block_overlap_rows.append({"block_a": i, "block_b": j, "length": L,
                                       "end_to_start_exact": eq})
display(pd.DataFrame(block_overlap_rows))

# Compressed nearest-neighbor audit. It is diagnostic, not automatically called
# leakage. Features preserve selected road levels, city moments, and differences.
def compressed_history_features(X):
    X = np.asarray(X, dtype=np.float32)
    probe = X[:, :, PROBE_ROADS].reshape(len(X), -1)
    city = np.stack([X.mean(2), X.std(2)], axis=2).reshape(len(X), -1)
    return np.concatenate([probe, city], axis=1).astype(np.float32)


def training_windows(arr, times):
    return np.stack([np.asarray(arr[t-HISTORY+1:t+1]) for t in times])


rng = np.random.default_rng(SEED)
nn_rows = []
test_cf = compressed_history_features(np.asarray(test_X))
for b, arr in enumerate(train_blocks):
    candidate_t = np.arange(HISTORY-1, len(arr)-max(HORIZONS),
                            2)
    W = training_windows(arr, candidate_t)
    F = compressed_history_features(W)
    mu, sd = F.mean(0), F.std(0) + 1e-4
    Fz, Qz = (F-mu)/sd, (test_cf-mu)/sd
    # Batch brute force avoids optional dependencies and a huge distance matrix.
    for q0 in range(0, N_TEST, 32):
        q = Qz[q0:q0+32]
        best_d = np.full(len(q), np.inf)
        best_i = np.full(len(q), -1)
        for f0 in range(0, len(Fz), 2048):
            f = Fz[f0:f0+2048]
            d = ((q[:, None, :] - f[None, :, :])**2).mean(2)
            ix = d.argmin(1)
            val = d[np.arange(len(q)), ix]
            take = val < best_d
            best_d[take], best_i[take] = val[take], f0 + ix[take]
        for k in range(len(q)):
            nn_rows.append({"test_sample": q0+k, "train_block": b,
                            "train_t": int(candidate_t[best_i[k]]),
                            "compressed_mse": float(best_d[k])})
    del W, F, Fz, Qz
nearest_audit = (pd.DataFrame(nn_rows).sort_values("compressed_mse")
                 .groupby("test_sample", as_index=False).first())
display(nearest_audit.describe())
display(nearest_audit.nsmallest(20, "compressed_mse"))


## 7. Evidence-backed implementation plan

EDA establishes four facts: traffic has strong low-rank city factors; daily grouping matters; graph neighbors are modestly more correlated than random roads; and raw text has weak road-level alignment. Therefore V7 uses:

1. fold-local whitened DirectLR with ranks 48/64 and V6's alpha 30;
2. V6's soft road-local residual persistence `(0.30, 0.15, 0.05)`;
3. a compact zero-initialized directed GNN trained only on normalized anchor residuals;
4. cross-fitted horizon damping and a completely locked audit fold;
5. one neural seed, no text, no FiLM, no MoE, and automatic fallback to the DirectLR anchor.

The success criterion is improvement on both three development folds and the locked audit, not a same-fold ensemble score.


In [ ]:
assert HISTORY==15 and HORIZONS==(5,10,15) and R==1260
RUN_STARTED=time.perf_counter()
def elapsed(): return time.perf_counter()-RUN_STARTED
def stage(name): print(f'[runtime] {name}: {elapsed()/60:.2f} min cumulative')
stage('data/audit')


## 8. Configuration and Kaggle T4 safeguards

In [ ]:
SEED=42
DIRECT_RANK_CANDIDATES=[48,64] if not FAST_MODE else [4,8]
DIRECT_ALPHA=30.0
LOCAL_RESIDUAL_SCALES=np.array([.30,.15,.05],dtype=np.float32)
CV_STRIDE=2 if not FAST_MODE else 30
FINAL_TRAIN_STRIDE=1 if not FAST_MODE else 35
BATCH_SIZE=16 if not FAST_MODE else 2
NUM_WORKERS=0
MAX_ABLATION_EPOCHS=5 if not FAST_MODE else 1
MAX_FINAL_EPOCHS=8 if not FAST_MODE else 1
EARLY_STOPPING_PATIENCE=2
ROAD_EMBED_DIM=12; TEMPORAL_HIDDEN=48; GRAPH_HIDDEN=48
DEV_FOLDS=[0,1,2]; LOCKED_AUDIT_FOLD=3; CHEAP_ANCHOR_FOLD=4

import importlib
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
if not hasattr(torch,'_utils'): torch._utils=importlib.import_module('torch._utils')
DEVICE=torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
torch.backends.cudnn.benchmark=DEVICE.type=='cuda'
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(SEED)
probe=nn.Parameter(torch.tensor([1.],device=DEVICE)); opt=torch.optim.AdamW([probe],lr=1e-3)
(probe.square().sum()).backward(); opt.step(); del probe,opt
AMP_OK=False
if DEVICE.type=='cuda':
    try:
        x=torch.randn(8,8,device=DEVICE); m=nn.Linear(8,8).to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16): m(x).sum().backward()
        AMP_OK=True; del x,m
    except Exception as exc: print('AMP preflight failed; FP32 fallback:',repr(exc))
print({'device':str(DEVICE),'amp':AMP_OK,'workers':NUM_WORKERS,'fast':FAST_MODE,
       'ranks':DIRECT_RANK_CANDIDATES,'alpha':DIRECT_ALPHA,'seeds':[SEED]})


## 9. Grouped-day development and locked-audit folds

Every history and its h15 target must remain inside one day. Folds 0/1/2 are development OOF, fold 3 is untouched until architecture and damping are fixed, and fold 4 is only an inexpensive anchor diagnostic. This is stricter than V5/V6's single randomized holdout.


In [ ]:
DAY=360
@dataclass(frozen=True)
class Ref:
    block:int;t:int;day:int
eligible=[]
for b,a in enumerate(train_blocks):
    for t in range(14,len(a)-15):
        if (t-14)//DAY==(t+15)//DAY:eligible.append(Ref(b,t,t//DAY))
day_fold={(b,d):d%5 for b,d in sorted(set((r.block,r.day) for r in eligible))}
def refs_for_fold(k,stride=1):
    tr=[r for r in eligible if day_fold[(r.block,r.day)]!=k][::stride]
    va=[r for r in eligible if day_fold[(r.block,r.day)]==k][::stride]
    assert set((r.block,r.day) for r in tr).isdisjoint(set((r.block,r.day) for r in va))
    assert all((r.t-14)//DAY==(r.t+15)//DAY for r in tr+va)
    return tr,va
fold_refs={k:refs_for_fold(k,CV_STRIDE) for k in range(5)}
for k,(tr,va) in fold_refs.items():
    print(k,len(tr),len(va),sorted(set(r.block for r in va)))
    assert set(r.block for r in va)==set(range(len(train_blocks)))
def get_hist(r):return np.asarray(train_blocks[r.block][r.t-14:r.t+1],np.float32)
def get_y(r):return np.stack([train_blocks[r.block][r.t+h] for h in HORIZONS],-1).astype(np.float32)

## 10. Fold-local V6 DirectLR anchor

The anchor forecasts all three horizons directly in whitened latent traffic space. Rank and alpha come from V6 evidence; only rank 48 versus 64 remains as a tiny development-fold comparison. The soft local residual restores high-frequency road detail removed by PCA, with decreasing strength at longer horizons.


In [ ]:
class DirectLRAnchor:
    def __init__(self,k,alpha=DIRECT_ALPHA): self.k=int(k); self.alpha=float(alpha)
    def fit(self,refs):
        train_days=set((r.block,r.day) for r in refs); rows=[]
        for b,a in enumerate(train_blocks):
            mask=np.array([(b,i//DAY) in train_days for i in range(len(a))])
            rows.append(np.asarray(a[mask],dtype=np.float32))
        Xrows=np.concatenate(rows)
        self.pca=PCA(n_components=self.k,whiten=True,svd_solver='randomized',random_state=SEED).fit(Xrows)
        self.rmean=Xrows.mean(0).astype(np.float32); self.rstd=(Xrows.std(0)+2).astype(np.float32)
        self.z=[self.pca.transform(np.asarray(a,dtype=np.float32)).astype(np.float32) for a in train_blocks]
        FX=[]; FY=[]
        for r in refs:
            zh=self.z[r.block][r.t-HISTORY+1:r.t+1]
            FX.append(zh.reshape(-1)); FY.append(np.concatenate([self.z[r.block][r.t+h] for h in HORIZONS]))
        self.model=Ridge(alpha=self.alpha).fit(np.asarray(FX,np.float32),np.asarray(FY,np.float32))
        del Xrows,rows,FX,FY; gc.collect(); return self
    def _pure_from_latent_history(self,zh):
        predz=self.model.predict(zh.reshape(1,-1)).reshape(3,self.k)
        return self.pca.inverse_transform(predz).astype(np.float32)
    def predict_pure(self,r):
        zh=self.z[r.block][r.t-HISTORY+1:r.t+1]
        return self._pure_from_latent_history(zh)
    def predict_anchor(self,r):
        H=get_hist(r); zh=self.z[r.block][r.t-HISTORY+1:r.t+1]
        pure=self._pure_from_latent_history(zh)
        local=H[-1]-self.pca.inverse_transform(zh[-1:])[0]
        return np.clip(pure+LOCAL_RESIDUAL_SCALES[:,None]*local[None],0,160).astype(np.float32)
    def test_predict(self,H):
        H=np.asarray(H,dtype=np.float32); zh=self.pca.transform(H)
        pure=self._pure_from_latent_history(zh)
        local=H[-1]-self.pca.inverse_transform(zh[-1:])[0]
        anchor=np.clip(pure+LOCAL_RESIDUAL_SCALES[:,None]*local[None],0,160)
        return pure.astype(np.float32),anchor.astype(np.float32)

def score(Y,P): return float(np.mean((Y.astype(np.float64)-P.astype(np.float64))**2))
anchor_rows=[]; anchor_models={}
folds_to_build=DEV_FOLDS+[LOCKED_AUDIT_FOLD,CHEAP_ANCHOR_FOLD]
for kfold in tqdm(folds_to_build,desc='Fold-local DirectLR anchors'):
    tr,va=fold_refs[kfold]; Y=np.stack([get_y(r).T for r in va])
    for dim in DIRECT_RANK_CANDIDATES:
        t0=time.perf_counter(); model=DirectLRAnchor(dim).fit(tr)
        pure=np.stack([model.predict_pure(r) for r in va])
        anchor=np.stack([model.predict_anchor(r) for r in va])
        anchor_rows += [dict(model=f'DLR{dim}_pure',fold=kfold,mse=score(Y,pure),runtime=time.perf_counter()-t0),
                        dict(model=f'DLR{dim}_resid_soft',fold=kfold,mse=score(Y,anchor),runtime=time.perf_counter()-t0)]
        anchor_models[(kfold,dim)]=(model,Y,pure,anchor,va)
anchor_df=pd.DataFrame(anchor_rows)
dev_mean=anchor_df[anchor_df.fold.isin(DEV_FOLDS)].groupby('model').mse.mean()
BEST_DIRECT_RANK=min(DIRECT_RANK_CANDIDATES,key=lambda d:dev_mean[f'DLR{d}_resid_soft'])
print('Selected DirectLR rank using development folds only:',BEST_DIRECT_RANK)
display(anchor_df.pivot(index='model',columns='fold',values='mse').round(5))
stage('DirectLR anchors')


## 11. Comparable natural-weight baselines

In [ ]:
baseline_rows=[]; baseline_day_rows=[]
for k in DEV_FOLDS+[LOCKED_AUDIT_FOLD]:
    model,Y,pure,anchor,va=anchor_models[(k,BEST_DIRECT_RANK)]
    candidates={
        'persistence':np.stack([np.repeat(get_hist(r)[-1][None],3,0) for r in va]),
        'mean3':np.stack([np.repeat(get_hist(r)[-3:].mean(0)[None],3,0) for r in va]),
        'DirectLR_pure':pure,'DirectLR_anchor':anchor,
    }
    for name,P in candidates.items():
        err=(Y.astype(np.float64)-P.astype(np.float64))**2
        hm=err.mean((0,2))
        baseline_rows.append({'model':name,'fold':k,'mse':err.mean(),'h5':hm[0],'h10':hm[1],'h15':hm[2]})
        for b,d in sorted(set((r.block,r.day) for r in va)):
            ix=np.array([(r.block,r.day)==(b,d) for r in va])
            baseline_day_rows.append({'model':name,'fold':k,'day':f'm{b+1}_d{d}','mse':err[ix].mean()})
baseline_df=pd.DataFrame(baseline_rows); baseline_day_df=pd.DataFrame(baseline_day_rows)
display(baseline_df.groupby('model')[['mse','h5','h10','h15']].mean().sort_values('mse').round(5))
stage('baselines')


## 12. Cached normalized residual dataset

The neural target is `truth − DirectLR anchor`, divided by a per-road/per-horizon residual scale estimated only from training-fold origins. This changes the task from forecasting the entire speed field to correcting a strong low-rank forecast.


In [ ]:
class ResidualDataset(Dataset):
    def __init__(self,refs,anchor_pred,res_scale,normalized_target=True):
        self.refs=refs; self.anchor=anchor_pred; self.scale=res_scale; self.normalized=normalized_target
    def __len__(self): return len(self.refs)
    def __getitem__(self,i):
        r=self.refs[i]; H=get_hist(r).T.copy(); P=self.anchor[i].T.copy(); Y=get_y(r)
        residual=Y-P
        if self.normalized: residual=residual/self.scale
        return torch.from_numpy(H),torch.from_numpy(P),torch.from_numpy(residual.astype(np.float32))

def build_fold_cache(k):
    model,Y,pure,anchor,va=anchor_models[(k,BEST_DIRECT_RANK)]; tr=fold_refs[k][0]
    train_anchor=np.stack([model.predict_anchor(r) for r in tqdm(tr,desc=f'fold{k} train anchors',leave=False)])
    sample=np.arange(0,len(tr),max(1,len(tr)//1200)); scales=[]
    for hi in range(3):
        rr=np.stack([get_y(tr[i])[:,hi]-train_anchor[i,hi] for i in sample])
        scales.append(np.clip(rr.std(0),2,20))
    scale=np.stack(scales,-1).astype(np.float32)
    return {'model':model,'tr':tr,'va':va,'train_p':train_anchor,'val_p':anchor,
            'val_pure':pure,'Y':Y,'scale':scale}


## 13. Compact zero-initialized directed residual-GNN

The ablation is intentionally narrow: temporal-only, road embeddings, and directed graph propagation. V7 does not retry text, FiLM, EMA, or MoE because Raphael's residual-model ablation rejected them and V6 already established the low-rank signal.


In [ ]:
src_np,dst_np=np.nonzero(np.asarray(A_raw)!=0); keep=src_np!=dst_np
src_np,dst_np=src_np[keep],dst_np[keep]
outdeg=np.bincount(src_np,minlength=R).clip(min=1).astype(np.float32)
indeg=np.bincount(dst_np,minlength=R).clip(min=1).astype(np.float32)

class DirectedBlock(nn.Module):
    def __init__(self,h):
        super().__init__(); self.ws=nn.Linear(h,h); self.wi=nn.Linear(h,h,bias=False); self.wo=nn.Linear(h,h,bias=False); self.norm=nn.LayerNorm(h)
    def forward(self,x,src,dst,ideg,odeg):
        incoming=torch.zeros_like(x); outgoing=torch.zeros_like(x)
        incoming.index_add_(1,dst,x[:,src]); outgoing.index_add_(1,src,x[:,dst])
        update=torch.nn.functional.gelu(self.ws(x)+self.wi(incoming/ideg[None,:,None])+self.wo(outgoing/odeg[None,:,None]))
        return self.norm(x+update)

@dataclass(frozen=True)
class NetCfg:
    name:str; embedding:bool=True; graph:bool=True

class DirectLRResidualGNN(nn.Module):
    def __init__(self,cfg,anchor):
        super().__init__(); self.cfg=cfg
        self.register_buffer('mean',torch.tensor(anchor.rmean)[None,:,None])
        self.register_buffer('std',torch.tensor(anchor.rstd)[None,:,None])
        self.register_buffer('src',torch.tensor(src_np,dtype=torch.long)); self.register_buffer('dst',torch.tensor(dst_np,dtype=torch.long))
        self.register_buffer('ideg',torch.tensor(indeg)); self.register_buffer('odeg',torch.tensor(outdeg))
        static=np.column_stack([anchor.rmean,anchor.rstd,out_degree,in_degree]).astype(np.float32)
        self.register_buffer('static',torch.tensor(static))
        self.temporal=nn.Sequential(nn.Conv1d(3,32,3,padding=1),nn.GELU(),nn.Conv1d(32,TEMPORAL_HIDDEN,3,padding=1),nn.GELU())
        self.emb=nn.Embedding(R,ROAD_EMBED_DIM) if cfg.embedding else None
        din=TEMPORAL_HIDDEN+3+4+(ROAD_EMBED_DIM if cfg.embedding else 0)
        self.fuse=nn.Sequential(nn.Linear(din,GRAPH_HIDDEN),nn.GELU())
        self.blocks=nn.ModuleList([DirectedBlock(GRAPH_HIDDEN),DirectedBlock(GRAPH_HIDDEN)]) if cfg.graph else nn.ModuleList()
        self.head=nn.Sequential(nn.Linear(GRAPH_HIDDEN,48),nn.GELU(),nn.Linear(48,3))
        nn.init.zeros_(self.head[-1].weight); nn.init.zeros_(self.head[-1].bias)
    def forward(self,H,P):
        norm=(H-self.mean)/self.std; dev=(H-H[:,:,-3:].mean(-1,keepdim=True))/self.std; zero=(H==0).to(H.dtype); B=H.shape[0]
        z=self.temporal(torch.stack([norm,dev,zero],2).reshape(B*R,3,HISTORY)).mean(-1).reshape(B,R,-1)
        parts=[z,P/self.std,self.static[None].expand(B,-1,-1)]
        if self.emb is not None: parts.append(self.emb(torch.arange(R,device=H.device))[None].expand(B,-1,-1))
        z=self.fuse(torch.cat(parts,-1))
        for block in self.blocks: z=block(z,self.src,self.dst,self.ideg,self.odeg)
        return self.head(z)


## 14. AMP-safe residual training and compact ablation

In [ ]:
def loader(ds,shuffle,seed):
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda',generator=g)

def predict_residual(model,ds,scale,desc='residual inference'):
    model.eval(); parts=[]
    with torch.no_grad():
        for H,P,_ in tqdm(loader(ds,False,SEED),desc=desc,leave=False):
            with torch.autocast(device_type=DEVICE.type,enabled=AMP_OK): q=model(H.to(DEVICE),P.to(DEVICE))
            parts.append(q.float().cpu().numpy())
    q=np.concatenate(parts).transpose(0,2,1)
    return q*scale.T[None] if ds.normalized else q

def train_net(cfg,cache,seed,epochs,normalized=True,val=True):
    set_seed(seed)
    trds=ResidualDataset(cache['tr'],cache['train_p'],cache['scale'],normalized)
    vds=ResidualDataset(cache['va'],cache['val_p'],cache['scale'],normalized)
    model=DirectLRResidualGNN(cfg,cache['model']).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
    scaler=torch.cuda.amp.GradScaler(enabled=AMP_OK); best=None; best_score=np.inf; stale=0; logs=[]
    bar=trange(1,epochs+1,desc=f'{cfg.name} seed={seed}',leave=False)
    for epoch in bar:
        model.train(); t0=time.perf_counter(); losses=[]
        for H,P,Y in loader(trds,True,seed+epoch):
            H,P,Y=H.to(DEVICE),P.to(DEVICE),Y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,enabled=AMP_OK):
                Q=model(H,P); loss=.9*torch.mean((Q-Y)**2)+.1*torch.nn.functional.smooth_l1_loss(Q,Y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),5)
            scaler.step(opt); scaler.update(); losses.append(float(loss.detach().cpu()))
        if val:
            pred=predict_residual(model,vds,cache['scale']); mse=score(cache['Y'],cache['val_p']+pred)
        else:
            pred=np.empty((0,3,R),np.float32); mse=float(np.mean(losses))
        logs.append({'epoch':epoch,'mse':mse,'train_loss':np.mean(losses),'seconds':time.perf_counter()-t0})
        bar.set_postfix(mse=f'{mse:.4f}')
        if not val:
            best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; best_score=mse; continue
        if mse<best_score-1e-4:
            best_score=mse; stale=0; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=EARLY_STOPPING_PATIENCE: break
    model.load_state_dict(best)
    if val: pred=predict_residual(model,vds,cache['scale'])
    return model,pred,pd.DataFrame(logs)

rep_cache=build_fold_cache(DEV_FOLDS[0])
base_cfg=NetCfg('temporal',False,False)
scale_trials=[]
for normalized in [False,True]:
    t0=time.perf_counter(); model,pred,log=train_net(base_cfg,rep_cache,SEED,MAX_ABLATION_EPOCHS,normalized)
    scale_trials.append({'normalized':normalized,'mse':score(rep_cache['Y'],rep_cache['val_p']+pred),'runtime':time.perf_counter()-t0})
    del model; collect_garbage()
display(pd.DataFrame(scale_trials).round(5))
NORMALIZED_TARGET=min(scale_trials,key=lambda x:x['mse'])['normalized']

ablation_specs=[NetCfg('temporal',False,False),NetCfg('road_embedding',True,False),NetCfg('directed_graph',True,True)]
ablation_rows=[]; best_cfg=base_cfg; best_mse=min(x['mse'] for x in scale_trials); best_epoch=MAX_ABLATION_EPOCHS
for cfg in ablation_specs:
    t0=time.perf_counter(); model,pred,log=train_net(cfg,rep_cache,SEED,MAX_ABLATION_EPOCHS,NORMALIZED_TARGET)
    mse=score(rep_cache['Y'],rep_cache['val_p']+pred)
    ablation_rows.append({'model':cfg.name,'fold':DEV_FOLDS[0],'mse':mse,'runtime':time.perf_counter()-t0})
    if mse<best_mse-.03:
        best_cfg,best_mse,best_epoch=cfg,mse,int(log.loc[log.mse.idxmin(),'epoch'])
    del model; collect_garbage()
ablation_df=pd.DataFrame(ablation_rows); display(ablation_df.round(5))
print('Selected residual architecture:',best_cfg,'| normalized target:',NORMALIZED_TARGET,'| epochs:',best_epoch)
stage('residual ablations')


## 15. Three-fold residual OOF and cross-fitted horizon damping

For each held development fold, beta is estimated only from the other two folds. This answers whether the neural correction magnitude transfers across days instead of merely fitting the same validation errors.


In [ ]:
dev_artifacts={}
for k in tqdm(DEV_FOLDS,desc='Development residual OOF'):
    cache=rep_cache if k==DEV_FOLDS[0] else build_fold_cache(k)
    t0=time.perf_counter(); model,residual,log=train_net(best_cfg,cache,SEED,best_epoch,NORMALIZED_TARGET)
    dev_artifacts[k]={'cache':cache,'res':residual,'log':log,'runtime':time.perf_counter()-t0}
    del model; collect_garbage()

betas={}; hybrid={}
for held in DEV_FOLDS:
    num=np.zeros(3); den=np.zeros(3)
    for k in DEV_FOLDS:
        if k==held: continue
        a=dev_artifacts[k]; error=a['cache']['Y']-a['cache']['val_p']; direction=a['res']
        num+=(error*direction).sum((0,2)); den+=(direction*direction).sum((0,2))
    beta=np.clip(num/np.maximum(den,1e-12),0,1.25); betas[held]=beta
    a=dev_artifacts[held]; hybrid[held]=a['cache']['val_p']+beta[None,:,None]*a['res']
FINAL_BETA=np.median(np.vstack(list(betas.values())),0)
print('Cross-fitted held-fold betas:',{k:np.round(v,4).tolist() for k,v in betas.items()})
print('Final median beta:',np.round(FINAL_BETA,4))
stage('development OOF')


## 16. Locked audit

Architecture, epoch count, normalization, DirectLR rank, and beta are frozen before fold 3 is opened. This is the notebook's most trustworthy local decision metric.


In [ ]:
audit_cache=build_fold_cache(LOCKED_AUDIT_FOLD)
audit_model,audit_res,audit_log=train_net(best_cfg,audit_cache,SEED,best_epoch,NORMALIZED_TARGET)
audit_hybrid=audit_cache['val_p']+FINAL_BETA[None,:,None]*audit_res
audit_anchor_mse=score(audit_cache['Y'],audit_cache['val_p'])
audit_hybrid_mse=score(audit_cache['Y'],audit_hybrid)
dev_anchor_mse=np.mean([score(dev_artifacts[k]['cache']['Y'],dev_artifacts[k]['cache']['val_p']) for k in DEV_FOLDS])
dev_hybrid_mse=np.mean([score(dev_artifacts[k]['cache']['Y'],hybrid[k]) for k in DEV_FOLDS])
print({'development_anchor':dev_anchor_mse,'development_hybrid':dev_hybrid_mse,
       'locked_anchor':audit_anchor_mse,'locked_hybrid':audit_hybrid_mse,
       'locked_gain':audit_anchor_mse-audit_hybrid_mse})
del audit_model; collect_garbage(); stage('locked audit')


## 17. Day robustness and acceptance gate

In [ ]:
daily=[]
for k in DEV_FOLDS:
    a=dev_artifacts[k]; Y=a['cache']['Y']; anchor=a['cache']['val_p']; pure=a['cache']['val_pure']; pred=hybrid[k]
    for b,d in sorted(set((r.block,r.day) for r in a['cache']['va'])):
        ix=np.array([(r.block,r.day)==(b,d) for r in a['cache']['va']])
        daily.append({'fold':k,'day':f'm{b+1}_d{d}','pure_mse':score(Y[ix],pure[ix]),
                      'anchor_mse':score(Y[ix],anchor[ix]),'hybrid_mse':score(Y[ix],pred[ix])})
daily_df=pd.DataFrame(daily)
win_rate=float((daily_df.hybrid_mse<daily_df.anchor_mse).mean())
dev_hybrid_h=np.mean([((dev_artifacts[k]['cache']['Y']-hybrid[k])**2).mean((0,2)) for k in DEV_FOLDS],0)
dev_anchor_h=np.mean([((dev_artifacts[k]['cache']['Y']-dev_artifacts[k]['cache']['val_p'])**2).mean((0,2)) for k in DEV_FOLDS],0)
block_ok=all(daily_df[daily_df.day.str.startswith(f'm{b}_')].hybrid_mse.mean()<=daily_df[daily_df.day.str.startswith(f'm{b}_')].anchor_mse.mean()+.10 for b in [1,2])
corr_mag=float(np.mean([np.abs(FINAL_BETA[None,:,None]*dev_artifacts[k]['res']).mean() for k in DEV_FOLDS]))
ACCEPT_NEURAL=(dev_hybrid_mse<dev_anchor_mse-.10 and audit_hybrid_mse<=audit_anchor_mse-.05
               and dev_hybrid_h[0]<dev_anchor_h[0] and dev_hybrid_h[2]<=dev_anchor_h[2]+.10
               and block_ok and win_rate>=.60 and corr_mag<10)
print({'accept':ACCEPT_NEURAL,'day_win_rate':win_rate,'mean_correction_kmh':corr_mag,
       'anchor_h':dev_anchor_h,'hybrid_h':dev_hybrid_h,'block_ok':block_ok})


## 18. Final full-data anchor and one-seed residual refit

V7 intentionally uses one neural seed. The final model is trained only if the locked acceptance gate passes; otherwise the notebook outputs the DirectLR anchor automatically.


In [ ]:
all_refs=eligible[::FINAL_TRAIN_STRIDE]
final_anchor=DirectLRAnchor(BEST_DIRECT_RANK).fit(all_refs)
final_train_p=np.stack([final_anchor.predict_anchor(r) for r in tqdm(all_refs,desc='Full DirectLR anchors')])
scale_ix=np.arange(0,len(all_refs),max(1,len(all_refs)//1500)); scales=[]
for hi in range(3):
    rr=np.stack([get_y(all_refs[i])[:,hi]-final_train_p[i,hi] for i in scale_ix]); scales.append(np.clip(rr.std(0),2,20))
final_scale=np.stack(scales,-1).astype(np.float32)
dummy={'model':final_anchor,'tr':all_refs,'va':[],'train_p':final_train_p,
       'val_p':np.empty((0,3,R),np.float32),'Y':np.empty((0,3,R),np.float32),'scale':final_scale}

test_pure=[]; test_anchor=[]
for H in tqdm(test_X,desc='DirectLR test inference'):
    p,a=final_anchor.test_predict(np.asarray(H,np.float32)); test_pure.append(p); test_anchor.append(a)
test_pure=np.asarray(test_pure); test_anchor=np.asarray(test_anchor)

test_residual=np.zeros_like(test_anchor); final_epochs=max(1,min(MAX_FINAL_EPOCHS,best_epoch))
if ACCEPT_NEURAL:
    model,_,final_log=train_net(best_cfg,dummy,SEED,final_epochs,NORMALIZED_TARGET,val=False)
    class TestDS(Dataset):
        normalized=NORMALIZED_TARGET
        def __len__(self): return N_TEST
        def __getitem__(self,i):
            return torch.from_numpy(np.asarray(test_X[i],np.float32).T.copy()),torch.from_numpy(test_anchor[i].T.copy()),torch.zeros((R,3))
    test_residual=predict_residual(model,TestDS(),final_scale,desc='V7 test residual')
    torch.save({'rank':BEST_DIRECT_RANK,'alpha':DIRECT_ALPHA,'config':asdict(best_cfg),'epochs':final_epochs,
                'beta':FINAL_BETA.tolist(),'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()}},OUTPUT_DIR/'v7_residual_gnn_seed42.pt')
    del model; collect_garbage()
test_hybrid=np.clip(test_anchor+FINAL_BETA[None,:,None]*test_residual,0,160)
test_final=test_hybrid if ACCEPT_NEURAL else test_anchor
stage('final refit and inference')


## 19. Results, visual evaluation, diagnostics, and submissions

In [ ]:
result_rows=[]
for k in DEV_FOLDS:
    a=dev_artifacts[k]; Y=a['cache']['Y']
    for name,P in [('anchor',a['cache']['val_p']),('hybrid_crossfit',hybrid[k])]:
        err=(Y-P)**2; hm=err.mean((0,2))
        result_rows.append({'model':name,'fold':k,'mse':err.mean(),'h5':hm[0],'h10':hm[1],'h15':hm[2]})
result_rows += [
    {'model':'anchor_locked','fold':LOCKED_AUDIT_FOLD,'mse':audit_anchor_mse},
    {'model':'hybrid_locked','fold':LOCKED_AUDIT_FOLD,'mse':audit_hybrid_mse},
]
results_df=pd.DataFrame(result_rows)
results_df.to_csv(OUTPUT_DIR/'v7_experiment_results.csv',index=False)
daily_df.to_csv(OUTPUT_DIR/'v7_daily_results.csv',index=False)
display(results_df.round(5)); display(daily_df.assign(gain=daily_df.anchor_mse-daily_df.hybrid_mse).describe().round(5))

ID_RE=re.compile(r'^test_(\d+)_h(5|10|15)_r(\d+)$'); horizon_map={5:0,10:1,15:2}
parsed=[ID_RE.fullmatch(str(x)) for x in sample.iloc[:,0]]; assert all(parsed)
ix=np.array([[int(m.group(1)),horizon_map[int(m.group(2))],int(m.group(3))] for m in parsed])
assert len(sample)==2041200 and sample.iloc[:,0].is_unique
def save_submission(pred,name):
    out=sample.copy(); out.iloc[:,1]=pred[ix[:,0],ix[:,1],ix[:,2]]
    assert len(out)==2041200 and np.isfinite(out.iloc[:,1]).all() and out.iloc[:,0].equals(sample.iloc[:,0])
    assert out.iloc[:,1].between(0,160).all(); out.to_csv(OUTPUT_DIR/name,index=False)
    print(name,{'mean':float(pred.mean()),'std':float(pred.std()),'min':float(pred.min()),'max':float(pred.max()),
                'mae_to_anchor':float(np.abs(pred-test_anchor).mean())})
    return out
anchor_submission=save_submission(test_anchor,'submission_v7_anchor.csv')
final_submission=save_submission(test_final,'submission_v7.csv')

if OPTIONAL['matplotlib']:
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(2,2,figsize=(15,9))
    results_df[results_df.fold.isin(DEV_FOLDS)].pivot(index='fold',columns='model',values='mse').plot(kind='bar',ax=ax[0,0],title='Development fold MSE')
    pd.DataFrame({'anchor':dev_anchor_h,'hybrid':dev_hybrid_h},index=[f'h{h}' for h in HORIZONS]).plot(kind='bar',ax=ax[0,1],title='Horizon MSE')
    ax[1,0].hist(daily_df.anchor_mse-daily_df.hybrid_mse,bins=25); ax[1,0].axvline(0,color='red',ls='--'); ax[1,0].set_title('Daily gain: anchor − hybrid')
    for i,h in enumerate(HORIZONS): ax[1,1].hist(test_final[:,i].ravel(),bins=60,alpha=.4,label=f'h{h}')
    ax[1,1].set_title('Submission distributions'); ax[1,1].legend(); plt.tight_layout(); plt.show()

diagnostics={'best_direct_rank':BEST_DIRECT_RANK,'direct_alpha':DIRECT_ALPHA,
             'residual_scales':LOCAL_RESIDUAL_SCALES.tolist(),'best_architecture':asdict(best_cfg),
             'normalized_residual':bool(NORMALIZED_TARGET),'epochs':final_epochs,'seed':SEED,
             'development_anchor_mse':float(dev_anchor_mse),'development_hybrid_mse':float(dev_hybrid_mse),
             'locked_anchor_mse':float(audit_anchor_mse),'locked_hybrid_mse':float(audit_hybrid_mse),
             'beta':FINAL_BETA.tolist(),'day_win_rate':win_rate,'accepted':bool(ACCEPT_NEURAL),
             'runtime_minutes':elapsed()/60,'recommended':'submission_v7.csv' if ACCEPT_NEURAL else 'submission_v7_anchor.csv'}
(OUTPUT_DIR/'v7_diagnostics.json').write_text(json.dumps(diagnostics,indent=2))
print('Saved V7 outputs to',OUTPUT_DIR); display(pd.concat([final_submission.head(3),final_submission.tail(3)]))
stage('outputs')


## 20. Final AI-readable decision report

In [ ]:
print('Best DirectLR rank:',BEST_DIRECT_RANK,'alpha:',DIRECT_ALPHA)
print('Residual architecture:',best_cfg,'| normalized:',NORMALIZED_TARGET,'| one seed:',SEED)
print('Development anchor → hybrid:',dev_anchor_mse,'→',dev_hybrid_mse,'gain',dev_anchor_mse-dev_hybrid_mse)
print('Locked audit anchor → hybrid:',audit_anchor_mse,'→',audit_hybrid_mse,'gain',audit_anchor_mse-audit_hybrid_mse)
print('Cross-fitted beta:',FINAL_BETA,'| daily win rate:',win_rate)
print('Accepted residual model:',ACCEPT_NEURAL)
print('Recommended submission:','submission_v7.csv' if ACCEPT_NEURAL else 'submission_v7_anchor.csv')
print('Runtime minutes:',elapsed()/60)
print('Interpretation: only a positive locked gain plus stable daily/horizon behavior authorizes the residual correction.')
